In [1]:
import json, os

def load_jsonl(path):
    with open(path, "r") as f:
        return [json.loads(line) for line in f]
    
# load data
dataset = load_jsonl("outputs/data/rubrics_8_10.jsonl")

Construct few-shot

In [2]:
# Generate few-shot examples based on high-frequency criteria
# This script selects rubrics with the most common criteria and creates few-shot examples.

from collections import Counter

# Read the data filtered in the previous step (rubrics quantity 8 to 10)
INPUT_PATH = "outputs/data/rubrics_8_10.jsonl"
OUTPUT_PATH = "outputs/data/few_shot_occurance.jsonl"
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

def load_jsonl(path):
    with open(path, "r") as f:
        return [json.loads(line) for line in f]

def save_jsonl(data, path):
    with open(path, "w") as f:
        for entry in data:
            json.dump(entry, f)
            f.write("\n")

# 1. Coiunt all criterion occurrences
all_criteria = []
for ex in dataset:
    for r in ex.get("rubrics", []):
        all_criteria.append(r["criterion"])
criterion_counter = Counter(all_criteria)

# 2. Select the top N most common criteria
TOP_N = 5  # 或10
top_criteria = set([c for c, _ in criterion_counter.most_common(TOP_N)])

# 3. Find rubrics samples containing these high-frequency criteria
candidate_rubrics = []
for ex in dataset:
    rubrics = ex.get("rubrics", [])
    prompt = ex.get("prompt", [])
    # Check if any high-frequency criterion is in the rubrics
    if any(r["criterion"] in top_criteria for r in rubrics):
        # Filter by conversation length
        total_chars = sum(len(turn.get("content", "")) for turn in prompt)
        if total_chars < 200:
            candidate_rubrics.append(ex)

# 4. For each rubric, select the top 3 most frequent criteria
few_shot_samples = []
for ex in candidate_rubrics:
    rubrics = ex.get("rubrics", [])
    # Sort by global occurrence count   
    rubrics_sorted = sorted(
        rubrics,
        key=lambda r: criterion_counter[r["criterion"]],
        reverse=True
    )
    # Select the top 3 rubrics
    selected_rubrics = rubrics_sorted[:3]
    few_shot_samples.append({
        "conversation": ex.get("prompt", []),
        "rubrics": selected_rubrics
    })

# 5. Save the results
import random
N_FEWSHOT = 3
final_fewshots = random.sample(few_shot_samples, min(N_FEWSHOT, len(few_shot_samples)))
save_jsonl(final_fewshots, OUTPUT_PATH)
print(f"✅ Saved {len(final_fewshots)} few-shot examples to {OUTPUT_PATH}")

✅ Saved 3 few-shot examples to outputs/data/few_shot_occurance.jsonl


In [9]:
from tqdm import tqdm

# Extract the prompt that needs to generate rubrics (only the prompt field, rubrics can be used as a reference)
INPUT_PATH = "outputs/data/rubrics_8_10_short.jsonl"
dataset = load_jsonl(INPUT_PATH)
target_conversations = [x for x in dataset if x.get("prompt") and isinstance(x["prompt"], list)]

# Save the first N conversations (adjustable)
N = len(target_conversations)

SAVE_DIR = "outputs/data/conversations_short"
os.makedirs(SAVE_DIR, exist_ok=True)

for i in tqdm(range(N), desc="Saving conversations"):
    conv = target_conversations[i]["prompt"]
    conv_text = "\n".join([f'{turn["role"].capitalize()}: {turn["content"]}' for turn in conv])
    with open(f"{SAVE_DIR}/conversation_{i}.txt", "w") as f:
        f.write(conv_text)

print(f"✅ Saved {N} conversations to {SAVE_DIR}")

Saving conversations: 100%|██████████| 254/254 [00:00<00:00, 27748.63it/s]

✅ Saved 254 conversations to outputs/data/conversations_short


RAG

Generate rubrics

In [ ]:
# ✅ log on Hugging Face（used to load model）
from huggingface_hub import login
import os
login(token=os.getenv("HF_TOKEN"))

/home/aychen/miniconda3/envs/rubrics/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from transformers import AutoTokenizer, pipeline, AutoModelForCausalLM

# set up the model
model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype="auto")
gen_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer)

Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.28it/s]
Device set to use cuda:0


In [5]:
# ✅ Prompt construction function
def build_prompt(conversation_path, reference_path, fewshot_path):
    with open(conversation_path) as f:
        target_conversation = f.read().strip()

    # Try to read the reference information (reference). If it cannot be found, mark it as empty
    if os.path.exists(reference_path):
        with open(reference_path) as f:
            reference = f.read().strip()
        reference_info = "Reference Info:\n" + reference + "\n\n"
    else:
        reference_info = "Reference Info:\n(No relevant Mayo Clinic reference was retrieved for this query.)\n\n"


    # 📌 Prompt header：Clearly distinguish between few-shot and the target task
    prompt = (
        "You are a medical assistant tasked with evaluating model responses in medical conversations.\n"
        "You will be given EXAMPLES of how to generate rubrics. Then, you will be asked to generate rubrics for a NEW conversation.\n\n"
        "Each rubric should:\n"
        "- contain a clear evaluation criterion (what to look for)\n"
        "- specify an axis: one of completeness, accuracy, context_awareness, communication_quality, instruction_following\n"
        "- assign a point between -10 and 10 (positive for good behavior, negative for harmful/incomplete info)\n\n"
        "=== FEW-SHOT EXAMPLES ===\n"
    )

    # 📌 Load few-shot example（At most a few rubrics）
    with open(fewshot_path) as f:
        fewshots = [json.loads(line) for line in f.readlines()]

    for i, example in enumerate(fewshots):
        prompt += f"\n=== EXAMPLE {i+1} ===\n"
        prompt += "Conversation:\n"
        for turn in example["conversation"]:
            prompt += f"{turn['role'].capitalize()}: {turn['content']}\n"
        prompt += "Rubrics:\n"
        for r in example["rubrics"]:
            axis = next((tag.split(":")[-1] for tag in r["tags"] if tag.startswith("axis:")), "unknown")
            point = r.get("points", 0)
            prompt += f"- Criterion: {r['criterion']}\n  Axis: {axis}\n  Point: {point}\n"

    # 📌 Target conversation & reference
    prompt += "\n=== TARGET CONVERSATION ===\n"
    prompt += "Conversation:\n" + target_conversation + "\n\n"
    prompt += reference_info

    # ✅ Insert the target generation instruction
    prompt += (
        "Please ensure the following when generating rubrics:\n\n"
        "- Generate **10 distinct criteria**, each rubric must cover **completeness, accuracy, and context_awareness** axes.\n"
        "- Include both **positive** and **negative** criteria.\n"
        "   - Positive rubrics: describe correct, helpful, or exemplary assistant behaviors (assign positive point values).\n"
        "   - Negative rubrics: describe missing, incorrect, misleading, harmful, or otherwise poor behaviors (assign negative point values, e.g., -1 to -10).\n"
        "- Each rubric must be **directly related to the specific conversation content**. Do not include generic or unrelated criteria.\n"
        "- Try to cover all five axes: completeness, accuracy, context_awareness, communication_quality, instruction_following\n"
        "Rubrics (in JSON list format):\n"
    )

    prompt += (
        "Now generate rubrics in JSON format as a list. Each item should include:\n"
        "- criterion (string)\n"
        "- axis (completeness | accuracy | context_awareness | communication_quality | instruction following)\n"
        "- point (integer between -10 and 10)\n\n"
        "Rubrics:\n"
    )

    return prompt


In [6]:
import re, json, os

# ✅ Output parsing: Extract rubrics JSON
def extract_rubrics_from_output(response):
    match = re.search(r"\[.*?\]", response, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except:
            pass

    # Fallback manual extraction
    rubrics = []
    current = {}
    for line in response.splitlines():
        if "Criterion:" in line:
            current["criterion"] = line.split("Criterion:")[-1].strip()
        if "Axis:" in line:
            current["axis"] = line.split("Axis:")[-1].strip().lower()
        if "Point:" in line:
            try:
                current["point"] = int(line.split("Point:")[-1].strip())
            except:
                current["point"] = 0
        if all(k in current for k in ("criterion", "axis", "point")):
            rubrics.append(current.copy())
            current = {}
    return rubrics


# ✅ Main function to run rubrics generation
def generate_rubrics(conversation_id):
    conversation_path = f"outputs/data/conversations_short/conversation_{conversation_id}.txt"
    reference_path = f"outputs/data/evidence/evidence_{conversation_id}.txt"
    fewshot_path = "outputs/data/few_shot_occurance.jsonl"  # use new short few-shot
    output_path = f"outputs/data_rubrics/gen_rubrics/rubrics_{conversation_id}.json"

    os.makedirs("outputs/data_rubrics/gen_rubrics/", exist_ok=True)

    prompt = build_prompt(conversation_path, reference_path, fewshot_path)

    print("\n📮 Prompt Preview:\n")
    print(prompt)

    # use LLM
    output = gen_pipeline(
        prompt,
        max_new_tokens=1024,
        do_sample=True,
        temperature=0.7
    )[0]["generated_text"]

    rubrics = extract_rubrics_from_output(output)

    # save
    with open(output_path, "w") as f:
        json.dump(rubrics, f, indent=2, ensure_ascii=False)

    print(f"\n✅ Saved generated rubrics to {output_path}")
    print("\n🔍 Conversation:\n")
    print(open(conversation_path).read())
    print("\n📌 First few rubrics:\n")
    print(json.dumps(rubrics[:3], indent=2, ensure_ascii=False))

In [7]:
# example
generate_rubrics(conversation_id=0) 

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



📮 Prompt Preview:

You are a medical assistant tasked with evaluating model responses in medical conversations.
You will be given EXAMPLES of how to generate rubrics. Then, you will be asked to generate rubrics for a NEW conversation.

Each rubric should:
- contain a clear evaluation criterion (what to look for)
- specify an axis: one of completeness, accuracy, context_awareness, communication_quality, instruction_following
- assign a point between -10 and 10 (positive for good behavior, negative for harmful/incomplete info)

=== FEW-SHOT EXAMPLES ===

=== EXAMPLE 1 ===
Conversation:
User: been feeling down is that depression
Rubrics:
- Criterion: The response has no factually incorrect information.
  Axis: accuracy
  Point: 5
- Criterion: The response seeks additional context to reduce uncertainty, either by asking follow-up questions or by suggesting what additional information would be helpful. The response should seek the most informative or useful context.
  Axis: context_awarene

In [8]:
from tqdm import tqdm

def batch_generate_rubrics(total_count):
    print(f"📦 Starting batch rubrics generation for {total_count} samples...")
    for i in tqdm(range(total_count), desc="Generating rubrics"):
        reference_path = f"outputs/data/evidence/evidence_{i}.txt"
        output_path = f"outputs/data_rubrics/gen_rubrics/rubrics_{i}.json"

        # If rubrics already exists, skip it (to avoid repeated generation)
        if os.path.exists(output_path):
            continue

        # If the reference does not exist, skip it (it will be automatically processed as a prompt without reference).
        if not os.path.exists(reference_path):
            print(f"⚠️ Missing reference for idx {i}, will generate without it.")

        try:
            generate_rubrics(conversation_id=i)
        except Exception as e:
            print(f"❌ Failed to generate rubrics for idx {i}: {e}")

In [ ]:
import os

CONVERSATION_DIR = "outputs/data/conversations_short"
total_count = len([
    f for f in os.listdir(CONVERSATION_DIR)
    if f.startswith("conversation_") and f.endswith(".txt")
])
batch_generate_rubrics(total_count=total_count)

Generating rubrics: 100%|██████████| 254/254 [2:17:31<00:00, 32.49s/it]


✅ Saved generated rubrics to outputs/data_rubrics/gen_rubrics/rubrics_253.json

🔍 Conversation:

User: Grandma started vomiting this morning

📌 First few rubrics:

[
  {}
]


: 

In [2]:
import os
import json

INPUT_DIR = "outputs/data_rubrics/gen_rubrics"
OUTPUT_PATH = "outputs/data_rubrics/gen_rubrics_all.jsonl"

files = [f for f in os.listdir(INPUT_DIR) if f.startswith("rubrics_") and f.endswith(".json")]
files = sorted(files, key=lambda x: int(x.split("_")[1].split(".")[0]))

with open(OUTPUT_PATH, "w", encoding="utf-8") as out_f:
    for fname in files:
        idx = int(fname.split("_")[1].split(".")[0])
        fpath = os.path.join(INPUT_DIR, fname)
        with open(fpath, "r", encoding="utf-8") as f:
            rubrics = json.load(f)
        if isinstance(rubrics, dict) and "rubrics" in rubrics:
            rubrics = rubrics["rubrics"]
        out_obj = {"index": idx, "rubrics": rubrics}
        out_f.write(json.dumps(out_obj, ensure_ascii=False) + "\n")

print(f"✅ Completed, saved to: {OUTPUT_PATH}")

✅ Completed, saved to: outputs/data_rubrics/gen_rubrics_all.jsonl
